In [3]:
import csv
import json
import re
from difflib import SequenceMatcher

def normalize_title(title):
    """
    Normalize movie title by removing common articles and punctuation.
    """
    # Remove articles and common words
    title = re.sub(r'\b(the|a|an)\b', '', title, flags=re.IGNORECASE)
    # Remove punctuation and extra spaces
    title = re.sub(r'[^\w\s]', '', title)
    # Remove extra whitespace
    title = re.sub(r'\s+', ' ', title).strip()
    return title.lower()

def extract_potential_titles(text):
    """
    Extract potential movie titles from text (quoted strings, capitalized phrases).
    """
    titles = []
    
    # Find quoted strings
    quoted_matches = re.findall(r'"([^"]*)"', text)
    titles.extend(quoted_matches)
    
    # Find phrases with QUOTATION_MARK format
    quotation_matches = re.findall(r'QUOTATION_MARK([^Q]+)QUOTATION_MARK', text)
    titles.extend(quotation_matches)
    
    # Find capitalized phrases (potential titles)
    capitalized_matches = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)
    titles.extend(capitalized_matches)
    
    return [title.strip() for title in titles if len(title.strip()) > 2]

def similarity_score(a, b):
    """Calculate similarity between two strings."""
    return SequenceMatcher(None, a, b).ratio()

def parse_ground_truth_movies(ground_truth):
    """
    Parse ground truth that may contain multiple movies separated by commas.
    Extract clean movie titles from entries like "Movie1 (year), Movie2 (year)"
    """
    movies = []
    # Split by comma and clean each title
    movie_parts = ground_truth.split(',')
    
    for part in movie_parts:
        part = part.strip()
        # Remove year in parentheses like "(1984)"
        clean_title = re.sub(r'\s*\(\d{4}\)\s*', '', part).strip()
        if clean_title:
            movies.append(clean_title)
    
    return movies

def check_movie_mention(ground_truth, conversation_text, similarity_threshold=0.7):
    """
    Check if any movie from ground truth is mentioned using multiple strategies:
    1. Direct substring match
    2. Normalized title match
    3. Partial/similarity match
    """
    # Parse multiple movies from ground truth
    ground_truth_movies = parse_ground_truth_movies(ground_truth)
    
    conversation_lower = conversation_text.lower()
    normalized_conversation = normalize_title(conversation_text)
    potential_titles = extract_potential_titles(conversation_text)
    
    # Check each movie in ground truth
    for movie in ground_truth_movies:
        movie_lower = movie.lower()
        normalized_movie = normalize_title(movie)
        
        # Strategy 1: Direct substring match
        if movie_lower in conversation_lower:
            return True, f"direct_match ({movie})"
        
        # # Strategy 2: Normalized title match
        # if normalized_movie in normalized_conversation:
        #     return True, f"normalized_match ({movie})"
        
        # # Strategy 3: Extract potential titles and check similarity
        # for potential_title in potential_titles:
        #     # Check direct match with potential title
        #     if similarity_score(movie_lower, potential_title.lower()) >= similarity_threshold:
        #         return True, f"similarity_match ({movie} -> {potential_title})"
            
        #     # Check normalized similarity
        #     if similarity_score(normalized_movie, normalize_title(potential_title)) >= similarity_threshold:
        #         return True, f"normalized_similarity_match ({movie} -> {potential_title})"
        
        # # Strategy 4: Check for partial matches (words from title)
        # movie_words = set(normalized_movie.split())
        # if len(movie_words) > 1:  # Only for multi-word titles
        #     conversation_words = set(normalized_conversation.split())
        #     # Check if most words from title appear in conversation
        #     word_overlap = len(movie_words.intersection(conversation_words))
        #     if word_overlap >= len(movie_words) * 0.75:  # 75% of words match
        #         return True, f"partial_word_match ({movie}: {word_overlap}/{len(movie_words)} words)"
    
    return False, "no_match"

def count_ground_truth_mentions(csv_file_path, similarity_threshold=0.7):
    """
    Count how many entries have the ground truth movie mentioned in the generated conversation.
    
    Args:
        csv_file_path (str): Path to the CSV file
        similarity_threshold (float): Similarity threshold for fuzzy matching (0-1)
        
    Returns:
        dict: Dictionary with counts and details
    """
    total_entries = 0
    successful_mentions = 0
    failed_entries = []
    match_types = {}
    
    try:
        with open(csv_file_path, 'r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            
            for row in reader:
                total_entries += 1
                ground_truth = row['ground_truth'].strip()
                generated_conversation = row['generated_conversation']
                
                # Parse the generated conversation JSON
                try:
                    conversation = json.loads(generated_conversation)
                    
                    # Convert conversation to a single string for searching
                    conversation_text = ""
                    for message in conversation:
                        if message.get('role') == 'assistant':
                            conversation_text += message.get('content', '') + " "
                    
                    # Check if ground truth movie is mentioned
                    is_mentioned, match_type = check_movie_mention(ground_truth, conversation_text, similarity_threshold)
                    
                    if is_mentioned:
                        successful_mentions += 1
                        match_types[match_type] = match_types.get(match_type, 0) + 1
                    else:
                        # Show which movies were being looked for in failed cases
                        movies_searched = parse_ground_truth_movies(ground_truth)
                        failed_entries.append({
                            'dialog_id': row['dialog_id'],
                            'ground_truth': ground_truth,
                            'movies_searched': movies_searched,
                            'reason': match_type
                        })
                        
                except json.JSONDecodeError:
                    print(f"Error parsing JSON for dialog_id: {row['dialog_id']}")
                    failed_entries.append({
                        'dialog_id': row['dialog_id'],
                        'ground_truth': ground_truth,
                        'error': 'JSON parsing error'
                    })
    
    except FileNotFoundError:
        print(f"Error: File '{csv_file_path}' not found.")
        return None
    except Exception as e:
        print(f"Error reading file: {str(e)}")
        return None
    
    # Calculate percentage
    success_rate = (successful_mentions / total_entries * 100) if total_entries > 0 else 0
    
    results = {
        'total_entries': total_entries,
        'successful_mentions': successful_mentions,
        'failed_mentions': total_entries - successful_mentions,
        'success_rate': success_rate,
        'failed_entries': failed_entries,
        'match_types': match_types
    }
    
    return results

def print_results(results):
    """Print the results in a formatted way."""
    if results is None:
        return
    
    print("=" * 60)
    print("GROUND TRUTH MENTION ANALYSIS (Enhanced)")
    print("=" * 60)
    print(f"Total entries: {results['total_entries']}")
    print(f"Successful mentions: {results['successful_mentions']}")
    print(f"Failed mentions: {results['failed_mentions']}")
    print(f"Success rate: {results['success_rate']:.2f}%")
    print()
    
    if results['match_types']:
        print("Match types breakdown:")
        for match_type, count in results['match_types'].items():
            percentage = (count / results['successful_mentions'] * 100)
            print(f"  - {match_type}: {count} ({percentage:.1f}%)")
        print()
    
    if results['failed_entries']:
        print("Failed entries (first 10):")
        for entry in results['failed_entries'][:10]:
            movies_info = f" (searched: {', '.join(entry['movies_searched'])})" if 'movies_searched' in entry else ""
            error_msg = f" (Error: {entry['error']})" if 'error' in entry else f" (Reason: {entry['reason']})"
            print(f"  - {entry['dialog_id']}: {entry['ground_truth']}{movies_info}{error_msg}")
        
        if len(results['failed_entries']) > 10:
            print(f"  ... and {len(results['failed_entries']) - 10} more")

# Example usage
if __name__ == "__main__":
    # Replace with your actual CSV file path
    csv_file_path = "../generated_testsets/multiturn_test/vanilla/llama-3.2-instruct/redial/generated_movie_conversations.csv"
    
    # You can adjust the similarity threshold (0.7 = 70% similarity)
    results = count_ground_truth_mentions(csv_file_path, similarity_threshold=0.9)
    print_results(results)

GROUND TRUTH MENTION ANALYSIS (Enhanced)
Total entries: 1076
Successful mentions: 424
Failed mentions: 652
Success rate: 39.41%

Match types breakdown:
  - direct_match (Police Academy): 1 (0.2%)
  - direct_match (It): 23 (5.4%)
  - direct_match (X-Men: The Last Stand): 1 (0.2%)
  - direct_match (The Mask): 2 (0.5%)
  - direct_match (The Exorcist): 2 (0.5%)
  - direct_match (Avengers: Infinity War): 4 (0.9%)
  - direct_match (The Conjuring): 3 (0.7%)
  - direct_match (Billy Madison): 1 (0.2%)
  - direct_match (A Quiet Place): 7 (1.7%)
  - direct_match (The Sixth Sense): 1 (0.2%)
  - direct_match (Grown Ups): 1 (0.2%)
  - direct_match (The Wedding Singer): 2 (0.5%)
  - direct_match (Black Sheep): 1 (0.2%)
  - direct_match (American Sniper): 2 (0.5%)
  - direct_match (A Nightmare on Elm Street): 1 (0.2%)
  - direct_match (Captain America: Civil War): 1 (0.2%)
  - direct_match (American Gangster): 3 (0.7%)
  - direct_match (Beauty and the Beast): 1 (0.2%)
  - direct_match (Blazing Saddles